In [ ]:
import os 
import time 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [ ]:
save_folder = 'run8'
n_points = 10000

n=3

lower_factor = 0.99
upper_factor = 2 - lower_factor

b_max = 30
q_max = 0.2



In [ ]:
# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)
x_totem, y_totem, yerr_totem = process_data(totem_data, totem_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]


In [ ]:

b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'eps': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'eps': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    }
}


# Get parameters for selected configuration
initial_params_log_atlas = ensemble_parameters['atlas']['log']
initial_params_pl_atlas = ensemble_parameters['atlas']['pl']



In [ ]:
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def born_amp(diff_T, s, epsilon, t):
    
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T


In [ ]:
# -------------------------------
# Integral over phi
# -------------------------------
def phi_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) - 
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    result, _ = fixed_quad(integrand, 0, 2*np.pi, n=n_points)
    return result

# -------------------------------
# Integral over k
# -------------------------------
def k_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    return phi_integral(k, mg, a1, a2, m2_func, q, n_points)

# -------------------------------
# Double integral computation over phi (first, inner) and k (second, outer)
# -------------------------------
def compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points=10000):
    result, _ = fixed_quad(
        lambda k: k_integral(k, mg, a1, a2, m2_func, q, n_points),
        0, sqrt_s_val, 
        n=n_points
    )
    return result


# -------------------------------
# Chi integral, over q
# -------------------------------
def chi_integral(sqrt_s_val, b, q_max, born_amp_value):
    s = sqrt_s_val**2
    
    # integrand over q
    def q_integrand(q):
        return (q * j0(b * q) * born_amp_value) / s
    
    # integrate real and imaginary parts separately with fixed_quad
    real_part, _ = fixed_quad(lambda q: np.real(np.vectorize(q_integrand)(q)), 0, q_max, n=n)
    imag_part, _ = fixed_quad(lambda q: np.imag(np.vectorize(q_integrand)(q)), 0, q_max, n=n)

    return real_part + 1j * imag_part


# -------------------------------
# Eikonal amplitude integral, over b
# -------------------------------
def eik_amp(sqrt_s_values, b_max, q_max, born_amp_value):
    # se for int ou float, transforma em lista
    if isinstance(sqrt_s_values, (int, float)):
        sqrt_s_values = [sqrt_s_values]

    amp_list = []
    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        def b_integrand(b):
            chi_val = chi_integral(sqrt_s_val, b, q_max, born_amp_value)
            return b * (1 - np.exp(1j * chi_val))
        
        real_part, _ = fixed_quad(lambda b: np.real(np.vectorize(b_integrand)(b)), 0, b_max, n=n)
        imag_part, _ = fixed_quad(lambda b: np.imag(np.vectorize(b_integrand)(b)), 0, b_max, n=n)
        
        A_eik = 1j * s * (real_part + 1j * imag_part)
        amp_list.append(A_eik)
    
    return amp_list if len(amp_list) > 1 else amp_list[0]



def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323




In [ ]:
#list for q integration for eikonal part
lst_q_eik = np.linspace(0, q_max, 130)

#chosing mass model type
mass_model = 'pl'
m2_func = m2_pl if mass_model == 'pl' else m2_log


# create a model function to use in least squares cost function
def model_function(x_born, eps, mg, a1, a2, sqrt_s, model_type='log'):
    # Choose mass function based on model type
    m2_func = m2_log if model_type == 'log' else m2_pl
    
    results = []
    s = sqrt_s ** 2
    
    # for loop for x experimental data points
    for q2 in x_born:
        t = -q2
        
        # Compute the double integral T1 and T2 using separate q's values, one for eikonal
        # and other related to experimental data
        for q_eik_val in lst_q_eik:
            diff_T = compute_double_integral(
                sqrt_s_val=sqrt_s,
                mg=mg,
                a1=a1,
                a2=a2,
                m2_func=m2_func,
                q=q_eik_val,
                n_points=n_points
            )
        
        # Calculate amplitudes
        born_amplitude = born_amp(diff_T, s, eps, t)
        eik_amplitude = eik_amp(s, b_max, q_max, born_amplitude)
        
        # Compute differential cross section
        diff_sigma = differential_sigma(eik_amplitude, s)
        results.append(diff_sigma)
    
    return np.array(results)

In [ ]:
def make_least_squares(x, y, yerr, energy, model_type):
    return LeastSquares(x, y, yerr, 
        lambda x, eps, mg, a1, a2: model_function(x, eps, mg, a1, a2, energy, model_type))

# Direct calculation without extra functions
total_cost_log_atlas = (
    make_least_squares(x_7_atlas, y_7_atlas, yerr_7_atlas, 7000, 'log') +
    make_least_squares(x_8_atlas, y_8_atlas, yerr_8_atlas, 8000, 'log') +
    make_least_squares(x_13_atlas, y_13_atlas, yerr_13_atlas, 13000, 'log')
)

total_cost_pl_atlas = (
    make_least_squares(x_7_atlas, y_7_atlas, yerr_7_atlas, 7000, 'pl') +
    make_least_squares(x_8_atlas, y_8_atlas, yerr_8_atlas, 8000, 'pl') +
    make_least_squares(x_13_atlas, y_13_atlas, yerr_13_atlas, 13000, 'pl')
)


In [ ]:
import time
from iminuit import Minuit
import gc
import numpy as np

def otimization(total_cost_func, initial_params, model_type: str, ensemble: str):
    
    print('\n' + 80 * '-')
    print(f"OTIMIZAÇÃO PARA {model_type.upper()} - {ensemble.upper()}")
    print(80 * '-' + '\n')
    
    if model_type == 'log' and ensemble == 'atlas':
        combo = {'mg': 1, 'eps': 0, 'a1': 1, 'a2': 1}  # Apenas eps livre, outros fixos
        down = 0.99
        strategy = 0
        ncall = 800
        tol = 1
        up = 2 - down

    elif model_type == 'pl' and ensemble == 'atlas':
        combo = {'mg': 0, 'eps': 0, 'a1': 1, 'a2': 1}  # Apenas eps livre, outros fixos
        down = 0.98
        strategy = 0
        ncall = 800
        tol = 5
        up = 2 - down
    else:
        print('Invalid model type or ensemble')


    print(f"Configuração de limites: eps_{combo['eps']}_mg_{combo['mg']}_a1_{combo['a1']}_a2_{combo['a2']}")
    print(f"Parâmetros: down={down}, up={up}, strategy={strategy}, tol={tol}, ncall={ncall}")
    print("=" * 100)

    start_time = time.time()

    # Verificação inicial da função de custo
    try:
        test_val = total_cost_func(
            initial_params['eps'],
            initial_params['mg'],
            initial_params['a1'],
            initial_params['a2']
        )
        if not np.isfinite(test_val):
            print("Valor inválido detectado! Abortando otimização.")
            return None
    except Exception as e:
        print(f"Erro na função de custo inicial: {e}")
        return None

    # Cria instância do Minuit
    m = Minuit(total_cost_func,
               eps=initial_params['eps'],
               mg=initial_params['mg'],
               a1=initial_params['a1'],
               a2=initial_params['a2'])

    # Aplica limites dinâmicos e fixação de parâmetros
    for param in ['mg', 'eps', 'a1', 'a2']:
        if combo[param] == 1:
            m.fixed[param] = True
        else:
            p0 = initial_params[param]  # Todos os parâmetros usam a mesma nomenclatura
            m.limits[param] = (p0 * down, p0 * up)

    # Configurações de otimização
    m.errordef = 1
    m.strategy = strategy
    m.tol = tol
    

    try:
        # Primeira etapa com parâmetros fixados
        print(">>> Primeira etapa: parâmetros fixados conforme configuração")
        m.simplex(ncall=ncall)
        m.migrad(ncall=ncall)

        # Segunda etapa: liberar todos os parâmetros
        m.fixed = False
        print(">>> Segunda etapa: todos parâmetros livres")
        m.migrad(ncall=ncall)
        m.migrad(ncall=ncall)
        
        m.hesse()
        m.minos(cl=0.9)

        execution_time = time.time() - start_time
        mins, secs = divmod(execution_time, 60)

        if m.valid:
            chi2_ndof = m.fval / m.ndof
            
            print('\n' + 80 * '-')
            print("RESULTADOS DA OTIMIZAÇÃO")
            print(80 * '-')
            print(f"Modelo: {model_type} | Ensemble: {ensemble}")
            print(f"\nParâmetros otimizados:")
            print(f"  mg  = {m.values['mg']:.6f} ± {m.errors['mg']:.6f}")
            print(f"  eps = {m.values['eps']:.6f} ± {m.errors['eps']:.6f}")
            print(f"  a1  = {m.values['a1']:.6f} ± {m.errors['a1']:.6f}")
            print(f"  a2  = {m.values['a2']:.6f} ± {m.errors['a2']:.6f}")
            print(f"\nEstatística:")
            print(f"  χ²/ndof = {chi2_ndof:.6f}")
            print(f"  Tempo de execução: {int(mins)} min {secs:.1f} s")
            print(80 * '-' + '\n')
            
            return m
        else:
            print("Minuit não convergiu para uma solução válida")
            return None
            
    except Exception as e:
        print(f"Erro durante otimização: {str(e)}")
        return None
    finally:
        # Libera memória explicitamente
        gc.collect()

In [ ]:
m_pl_atlas = otimization(total_cost_pl_atlas,
                         initial_params_pl_atlas, 'pl', 'atlas')
